In [1]:
# script_pncp/classificar_bart_local.py
import argparse
import sqlite3
import json
from pathlib import Path
from typing import List

# importe componentes do seu pacote de modelo
from modelo_classificacao_relevancia.app.config.settings import AppSettings
from modelo_classificacao_relevancia.app.services.bart_model_service import BartModelService
from modelo_classificacao_relevancia.app.services.text_cleaner import TextCleaner
from modelo_classificacao_relevancia.app.services.rule_engine import RuleEngine
from modelo_classificacao_relevancia.app.services.decision_service import DecisionService
from modelo_classificacao_relevancia.app.services.classification_service import ClassificationService

def obter_objeto_compra(row: dict) -> str:
    # mesma prioridade: coluna direta -> dados_json.objetoCompra (ou variantes)
    if row.get("objeto_compra"):
        return row["objeto_compra"]
    # tente diferentes chaves no dados_json
    raw = row.get("dados_json") or row.get("dados") or ""
    if raw:
        try:
            d = json.loads(raw) if isinstance(raw, str) else raw
            return d.get("objetoCompra") or d.get("objeto_compra") or d.get("Objeto") or ""
        except Exception:
            return ""
    return ""

def criar_tabela_saida(conn: sqlite3.Connection):
    conn.execute(
        """CREATE TABLE IF NOT EXISTS contratacoes_filtradas (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            pncp_id TEXT UNIQUE,
            numero_controle TEXT,
            objeto_compra TEXT,
            prob_nao_relevante REAL,
            prob_relevante REAL,
            categoria TEXT,
            motivo TEXT,
            origem TEXT,
            criado_em TEXT DEFAULT (datetime('now'))
        )"""
    )
    conn.commit()

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--db", required=True, help="DB de origem")
    parser.add_argument("--out", required=True, help="DB de saída")
    parser.add_argument("--model-dir", default=None, help="Dir do modelo BART")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--limit", type=int, default=None)
    args = parser.parse_args()

    # Configurações do app de modelo
    settings = AppSettings()
    if args.model_dir:
        # override simples
        settings = AppSettings(model=type(settings.model)(model_dir=args.model_dir, max_length=settings.model.max_length))

    model_service = BartModelService(settings=settings.model)
    cleaner = TextCleaner()
    rule_engine = RuleEngine(cleaner=cleaner, settings=settings.rules)
    decision_service = DecisionService(thresholds=settings.thresholds)
    classifier = ClassificationService(cleaner=cleaner, rule_engine=rule_engine, model_service=model_service, decision_service=decision_service)

    # Conexões DB
    conn_src = sqlite3.connect(args.db)
    conn_src.row_factory = sqlite3.Row
    conn_out = sqlite3.connect(args.out)
    criar_tabela_saida(conn_out)

    # Leitura: adapte o SELECT à sua tabela (contratacoes / contratacoes_filtradas)
    rows = list(conn_src.execute("SELECT * FROM contratacoes ORDER BY id" + (f" LIMIT {args.limit}" if args.limit else "")).fetchall())
    texts = []
    mapping = []  # guarda (pncp_id, numero_controle, objeto_text)
    for r in rows:
        row = dict(r)
        objeto = obter_objeto_compra(row) or ""
        pncp = row.get("pncp_id") or row.get("numeroControlePNCP") or ""
        numero_controle = row.get("pncp_id") or pncp
        texts.append(objeto)
        mapping.append((pncp, numero_controle, objeto))

    # Processa em batches
    batch_size = args.batch_size
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        probs = model_service.predict_probabilities(batch_texts)  # retorna [[p_nao, p_sim], ...]
        # opcional: use classifier.classify(batch_texts) para aplicar regras/decision_service e retornar dataframe
        df = classifier.classify(batch_texts)
        # salvar resultados
        for j, row_res in df.iterrows():
            pncp_id = mapping[i + j][0]
            numero_controle = mapping[i + j][1]
            objeto_text = mapping[i + j][2]
            prob_nao = row_res["prob_nao_relevante"]
            prob_relev = row_res["prob_relevante"]
            categoria = row_res.get("classificacao_final", "")  # depende de como DecisionService popula
            motivo = row_res.get("motivo_regra", "") or row_res.get("status_final", "")
            origem = row_res.get("origem_decisao", "")
            conn_out.execute(
                """INSERT OR REPLACE INTO contratacoes_filtradas
                (pncp_id, numero_controle, objeto_compra, prob_nao_relevante, prob_relevante, categoria, motivo, origem)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
                (pncp_id, numero_controle, objeto_text, prob_nao, prob_relev, categoria, motivo, origem),
            )
        conn_out.commit()
        print(f"Batch {i // batch_size + 1} / {(len(texts)-1)//batch_size + 1} salvo.")

    conn_src.close()
    conn_out.close()
    print("Classificação concluída. DB salvo em:", args.out)

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'modelo_classificacao_relevancia'